# Development cross-validation

Repeated stratified cross-validation is performed on the development data only. The temporal test set remains untouched for final evaluation. Reusable logic lives in `icare_risk.evaluation.cross_validation`.

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from pathlib import Path
import yaml
from IPython.display import display

from icare_risk.data.audit import load_admission_file
from icare_risk.evaluation.cross_validation import cross_validate_models

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

df = load_admission_file(config['outputs']['processed_dataset'])
development = df.loc[df['split'].isin(['train', 'validation'])].copy()
target = config['columns']['target']
target_mapping = config.get('target_mapping', config.get('analysis', {}).get('target_mapping', {}))
print(f'Development admissions: {len(development):,}')

Development admissions: 7,517


In [9]:
cv_cfg = config.get('cross_validation', {})
fold_results, cv_summary = cross_validate_models(
    development,
    config['models'],
    config['feature_sets'],
    target=target,
    target_mapping=target_mapping,
    exclude_columns=[target, 'split'],
    n_splits=cv_cfg.get('n_splits', 5),
    n_repeats=cv_cfg.get('n_repeats', 3),
    threshold=config.get('analysis', {}).get('evaluation_threshold', 0.5),
    random_state=cv_cfg.get('random_state', 42),
)
display(cv_summary.sort_values(['metric', 'median']))
display(fold_results.head())

,model,n_features,metric,n_folds,median,q1,q3,iqr,mean,sd
22,xgb_kim_score,5,auprc,15,0.192295,0.191545,0.193779,0.002234,0.192251,0.001868
1,lr_kim_score,5,auprc,15,0.192389,0.191545,0.193779,0.002234,0.192298,0.001862
8,lr_all_scores,6,auprc,15,0.192610,0.189678,0.196729,0.007051,0.192804,0.006117
29,xgb_all_scores,6,auprc,15,0.196747,0.192231,0.203180,0.010949,0.197171,0.007321
36,xgb_clinical_baseline,16,auprc,15,0.198177,0.194111,0.202416,0.008304,0.198412,0.005920
15,lr_clinical_baseline,16,auprc,15,0.198955,0.188667,0.204502,0.015835,0.197226,0.008925
21,xgb_kim_score,5,auroc,15,0.500453,0.498630,0.505870,0.007240,0.501021,0.006778
0,lr_kim_score,5,auroc,15,0.500755,0.499099,0.505870,0.006771,0.501174,0.006742
35,xgb_clinical_baseline,16,auroc,15,0.505360,0.499476,0.510595,0.011119,0.504701,0.008316
14,lr_clinical_baseline,16,auroc,15,0.505645,0.489040,0.516336,0.027296,0.504152,0.018576


,model,n_features,repeat,fold,n_train,n_validation,model_type,n,n_events,event_rate,...,tn,fp,fn,sensitivity,specificity,ppv,npv,accuracy,balanced_accuracy,f1
0,lr_kim_score,5,1,1,6013,1504,logistic_regression,1504,289,0.192154,...,147,1068,31,0.892734,0.120988,0.194570,0.825843,0.269282,0.506861,0.319505
1,lr_kim_score,5,1,2,6013,1504,logistic_regression,1504,289,0.192154,...,118,1097,31,0.892734,0.097119,0.190406,0.791946,0.250000,0.494926,0.313869
2,lr_kim_score,5,1,3,6014,1503,logistic_regression,1503,288,0.191617,...,129,1086,26,0.909722,0.106173,0.194362,0.832258,0.260146,0.507948,0.320293
3,lr_kim_score,5,1,4,6014,1503,logistic_regression,1503,288,0.191617,...,136,1079,28,0.902778,0.111934,0.194175,0.829268,0.263473,0.507356,0.319607
4,lr_kim_score,5,1,5,6014,1503,logistic_regression,1503,288,0.191617,...,135,1080,32,0.888889,0.111111,0.191617,0.808383,0.260146,0.500000,0.315271


In [ ]:
output_directory = Path(config.get('outputs', {}).get('metrics_directory', 'results/metrics'))
output_directory.mkdir(parents=True, exist_ok=True)
fold_results.to_csv(output_directory / 'cross_validation_folds.csv', index=False)
cv_summary.to_csv(output_directory / 'cross_validation_summary.csv', index=False)
print(f'Saved cross-validation results to {output_directory}')